# Ejemplo 2 — Sintonía PI mediante Root Locus

En este cuaderno se verifica el diseño analítico de un controlador PI para la planta

$$G(s)=\frac{1}{(s+1)(s+2)(s+10)}.$$

El controlador utilizado es

$$C_{PI}(s)=K\frac{s+0.1}{s}.$$

El cero se ubicó en $s=-0.1$ y se mantuvo como especificación de diseño

$$\zeta=0.5.$$

A partir del desarrollo analítico de la lectura se obtuvo la ecuación

$$129\rho^3-320\rho^2+32\rho-20=0,$$

cuyo valor real positivo es aproximadamente

$$\rho\approx2.4043.$$

Con este valor se verificará la ubicación del polo deseado, la ganancia $K$, los polos de lazo cerrado y el error de estado estacionario.

In [ ]:
!pip -q install control

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import control as ct

## Definición de la planta y del controlador PI

Se define la planta y se fija la ubicación del cero integral en $s=-0.1$.

In [ ]:
s = ct.TransferFunction.s

G = 1 / ((s + 1)*(s + 2)*(s + 10))

zI = 0.1

G_PI = (s + zI) / (s*(s + 1)*(s + 2)*(s + 10))

print('Planta:')
print(G)

print('\nFunción de lazo abierto sin incluir K:')
print(G_PI)

## Verificación del valor de $\rho$

La ecuación obtenida en la lectura es

$$129\rho^3-320\rho^2+32\rho-20=0.$$

Calculamos sus raíces y seleccionamos la raíz real positiva.

In [ ]:
coeficientes = [129, -320, 32, -20]
raices_rho = np.roots(coeficientes)

print('Raíces de la ecuación en rho:')
for raiz in raices_rho:
    print(raiz)

rho = np.real(raices_rho[np.argmin(np.abs(np.imag(raices_rho)))])

print(f'\nrho = {rho:.6f}')

## Polo deseado

Para $\zeta=0.5$,

$$\theta=\cos^{-1}(0.5)=60^\circ.$$

Por tanto,

$$s_d=-\rho\cos(60^\circ)+j\rho\sin(60^\circ).$$

In [ ]:
zeta = 0.5
theta = np.arccos(zeta)

sd = -rho*np.cos(theta) + 1j*rho*np.sin(theta)

print(f'sd = {sd.real:.6f} {sd.imag:+.6f}j')

## Cálculo de la ganancia $K$

Para el controlador PI,

$$L(s)=K\frac{s+0.1}{s(s+1)(s+2)(s+10)}.$$

La condición de magnitud proporciona

$$K=\frac{|s_d|\,|s_d+1|\,|s_d+2|\,|s_d+10|}{|s_d+0.1|}.$$

In [ ]:
K = abs(sd*(sd + 1)*(sd + 2)*(sd + 10)) / abs(sd + zI)

print(f'K = {K:.6f}')

## Ganancias equivalentes del controlador PI

Como

$$C_{PI}(s)=K\frac{s+z_I}{s}=K+\frac{Kz_I}{s},$$

las ganancias equivalentes son

$$K_P=K,\qquad K_I=Kz_I.$$

In [ ]:
KP = K
KI = K*zI

print(f'K_P = {KP:.6f}')
print(f'K_I = {KI:.6f}')

## Polos de lazo cerrado

Se construye el controlador PI con la ganancia obtenida y se verifican los polos del sistema de lazo cerrado.

In [ ]:
C_PI = K*(s + zI)/s
T_PI = ct.feedback(C_PI*G, 1)

polos_PI = ct.poles(T_PI)

print('Polos del sistema de lazo cerrado:')
for polo in polos_PI:
    print(f'{polo.real:.6f} {polo.imag:+.6f}j')

## Root Locus

El punto calculado analíticamente se marca sobre el Root Locus del sistema con compensación PI.

In [ ]:
plt.figure(figsize=(8, 6))
ct.root_locus(G_PI, grid=True)

plt.plot(sd.real, sd.imag, 'rx', markersize=10, markeredgewidth=2,
         label='Polos deseados')
plt.plot(sd.real, -sd.imag, 'rx', markersize=10, markeredgewidth=2)

plt.xlabel('Parte real')
plt.ylabel('Parte imaginaria')
plt.title('Root Locus con controlador PI')
plt.legend()
plt.grid(True)
plt.show()

## Respuesta al escalón

Se observa la respuesta al escalón del sistema compensado. El integrador aumenta el tipo del sistema y permite eliminar el error de estado estacionario para una entrada escalón.

In [ ]:
t, y = ct.step_response(T_PI)

plt.figure(figsize=(8, 5))
plt.plot(t, y, label='Control PI')
plt.axhline(1, linestyle='--', linewidth=1, label='Referencia')

plt.xlabel('Tiempo [s]')
plt.ylabel('Salida')
plt.title('Respuesta al escalón')
plt.grid(True)
plt.legend()
plt.show()

## Error de estado estacionario

El controlador PI introduce un polo en el origen. Por tanto, para una entrada escalón, el sistema es de tipo 1 y se espera

$$e_{ss}=0.$$

In [ ]:
valor_final = ct.dcgain(T_PI)
ess = 1 - valor_final

print(f'Valor final de la salida = {valor_final:.6f}')
print(f'Error de estado estacionario = {ess:.6e}')